Cleaning for Ppro

In [2]:
import pandas as pd
import re

In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd().parent  
RAW = BASE_DIR / "Data" / "raw"
# Load the datasets
df = pd.read_csv(RAW / "ppro_raw_listings.csv")

In [ ]:

df.head()

,page,Price,Location,Bedrooms,List Date
0,1,"₦ 35,000,000/year",Diplomatic Zone Asokoro Abuja,Furnished 5 Bedroom Detached Duplex,"Updated 09 Feb 2026, Added 01 Jan 2026"
1,1,"₦ 40,000,000/year",Asokoro District Abuja Diplomatic Zone Asokoro...,6 Bedroom Mansion With 2room Bq,"Updated 09 Feb 2026, Added 22 Dec 2025"
2,1,"₦ 13,000,000/year",Asokoro Abuja Diplomatic Zone Asokoro Abuja,3 Bedroom Block Of Flat,"Updated 09 Feb 2026, Added 11 Dec 2025"
3,1,"₦ 18,000,000/year",Guzape District Abuja Diplomatic Zone Guzape A...,4 Bedroom Terrace Duplex With Bq,"Updated 09 Feb 2026, Added 22 Dec 2025"
4,1,"₦ 25,000,000/year","Diplomatic Zone, Guzape Abuja",Newly Built 4 Bedroom Terrace Duplex With Bq,"Updated 09 Feb 2026, Added 10 Dec 2025"


In [4]:
df = df.drop(['page', 'List Date'], axis=1)
df.head()

,Price,Location,Bedrooms
0,"₦ 35,000,000/year",Diplomatic Zone Asokoro Abuja,Furnished 5 Bedroom Detached Duplex
1,"₦ 40,000,000/year",Asokoro District Abuja Diplomatic Zone Asokoro...,6 Bedroom Mansion With 2room Bq
2,"₦ 13,000,000/year",Asokoro Abuja Diplomatic Zone Asokoro Abuja,3 Bedroom Block Of Flat
3,"₦ 18,000,000/year",Guzape District Abuja Diplomatic Zone Guzape A...,4 Bedroom Terrace Duplex With Bq
4,"₦ 25,000,000/year","Diplomatic Zone, Guzape Abuja",Newly Built 4 Bedroom Terrace Duplex With Bq


In [5]:
#Create Property type column from messy 'Bedrooms' column and clean 'Bedrooms' column
df['Bedrooms(s)'] = df['Bedrooms'].str.extract(r'(\d+)')
df['Property Type'] = df['Bedrooms'].str.replace(r'\d+\s*Bedroom\s*', '', regex=True).str.strip()
df.head()

,Price,Location,Bedrooms,Bedrooms(s),Property Type
0,"₦ 35,000,000/year",Diplomatic Zone Asokoro Abuja,Furnished 5 Bedroom Detached Duplex,5,Furnished Detached Duplex
1,"₦ 40,000,000/year",Asokoro District Abuja Diplomatic Zone Asokoro...,6 Bedroom Mansion With 2room Bq,6,Mansion With 2room Bq
2,"₦ 13,000,000/year",Asokoro Abuja Diplomatic Zone Asokoro Abuja,3 Bedroom Block Of Flat,3,Block Of Flat
3,"₦ 18,000,000/year",Guzape District Abuja Diplomatic Zone Guzape A...,4 Bedroom Terrace Duplex With Bq,4,Terrace Duplex With Bq
4,"₦ 25,000,000/year","Diplomatic Zone, Guzape Abuja",Newly Built 4 Bedroom Terrace Duplex With Bq,4,Newly Built Terrace Duplex With Bq


In [6]:
df = df.drop(['Bedrooms'], axis = 1)
df.head()

,Price,Location,Bedrooms(s),Property Type
0,"₦ 35,000,000/year",Diplomatic Zone Asokoro Abuja,5,Furnished Detached Duplex
1,"₦ 40,000,000/year",Asokoro District Abuja Diplomatic Zone Asokoro...,6,Mansion With 2room Bq
2,"₦ 13,000,000/year",Asokoro Abuja Diplomatic Zone Asokoro Abuja,3,Block Of Flat
3,"₦ 18,000,000/year",Guzape District Abuja Diplomatic Zone Guzape A...,4,Terrace Duplex With Bq
4,"₦ 25,000,000/year","Diplomatic Zone, Guzape Abuja",4,Newly Built Terrace Duplex With Bq


In [ ]:
BASE_DIR = Path.cwd().parent  

DATA = BASE_DIR / "Data"
df.to_csv(DATA / "messy.csv", index=False)

In [8]:
import numpy as np

# 1. Create the categorization logic
def prop_category(text):
    # Convert everything to lowercase to make matching easier
    text = str(text).lower()
    
    # Priority 1: Identify and flag non-residential or unwanted properties first
    non_residential_keywords = ['office', 'warehouse', 'shop', 'land', 'commercial', 'plaza', 'store', 'hotel']
    if any(word in text for word in non_residential_keywords):
        return np.nan  # Tag as NaN so we can drop it easily later

    # Priority 2: Categorize as Bungalow
    if 'bungalow' in text:
        return 'Bungalow'

    # Priority 3: Categorize as Duplex 
    # (Including 'mansion' and 'terrace' here since they function as duplexes in this market)
    duplex_keywords = ['duplex', 'mansion', 'terrace', 'penthouse']
    if any(word in text for word in duplex_keywords):
        return 'Duplex'

    # Priority 4: Categorize as Apartment
    apartment_keywords = ['flat', 'apartment', 'room', 'studio']
    if any(word in text for word in apartment_keywords):
        return 'Apartment'

    # Catch-all: If it doesn't match anything above, flag it to be dropped
    return np.nan

# 2. Apply the function to create the new column
df['Property Category'] = df['Property Type'].apply(prop_category)

# 3. Drop all rows that were tagged as non-residential or unmatched (NaN)
df_cleaned = df.dropna(subset=['Property Category']).reset_index(drop=True)

# Let's check the results
print(df_cleaned[['Property Type', 'Property Category']].head())

                        Property Type Property Category
0           Furnished Detached Duplex            Duplex
1               Mansion With 2room Bq            Duplex
2                       Block Of Flat         Apartment
3              Terrace Duplex With Bq            Duplex
4  Newly Built Terrace Duplex With Bq            Duplex


In [18]:
df.head()

,Price,Location,Bedrooms(s),Property Type,Property Category,District
0,"₦ 35,000,000/year",Diplomatic Zone Asokoro Abuja,5,Furnished Detached Duplex,Duplex,Asokoro
1,"₦ 40,000,000/year",Asokoro District Abuja Diplomatic Zone Asokoro...,6,Mansion With 2room Bq,Duplex,Asokoro
2,"₦ 13,000,000/year",Asokoro Abuja Diplomatic Zone Asokoro Abuja,3,Block Of Flat,Apartment,Asokoro
3,"₦ 18,000,000/year",Guzape District Abuja Diplomatic Zone Guzape A...,4,Terrace Duplex With Bq,Duplex,Guzape
4,"₦ 25,000,000/year","Diplomatic Zone, Guzape Abuja",4,Newly Built Terrace Duplex With Bq,Duplex,Guzape


In [13]:
df.shape

(2155, 5)

In [25]:
# This gives you a list of every column and the count of missing values
empty_counts = df.isnull().sum()
print(empty_counts)

Price                  0
Location               0
Bedrooms(s)          415
Property Type          0
Property Category    295
District               0
Price_Cleaned          0
dtype: int64


In [16]:
#create district column 
amac_districts = ['Jabi', 'Kaura', 'Garki', 'Kabusa', 'City Centre', 'Wuse', 'Gwarinpa', 'Gui', 'Karshi', 'Asokoro', 'Jahi', 'Guzape', 'Apo', 'Durumi', 'Lugbe', 'Lokogoma', 'Maitama', 'Wuye', 'Katampe', 'Life Camp', 'Utako', 'Mabushi', 'Idu', 'Kado']

# Extract district name from Location
def extract_district(location):
    if pd.isna(location):
        return np.nan
    for district in amac_districts:
        if district.lower() in location.lower():
            return district
    return np.nan

df["District"] = df["Location"].apply(extract_district)

#Drop rows where District is NaN (not in AMAC list)
df = df.dropna(subset=["District"])

df[["Location", "District"]].head(10)

,Location,District
0,Diplomatic Zone Asokoro Abuja,Asokoro
1,Asokoro District Abuja Diplomatic Zone Asokoro...,Asokoro
2,Asokoro Abuja Diplomatic Zone Asokoro Abuja,Asokoro
3,Guzape District Abuja Diplomatic Zone Guzape A...,Guzape
4,"Diplomatic Zone, Guzape Abuja",Guzape
5,Asokoro District Abuja Diplomatic Zone Asokoro...,Asokoro
6,"Diplomatic Zone, Guzape Abuja",Guzape
7,Guzape District Abuja Diplomatic Zone Guzape A...,Guzape
8,"Asokoro District Abuja Diplomatic Zone, Asokor...",Asokoro
10,Navy Town Estate Asokoro Diplomatic Zone In As...,Asokoro


In [24]:
#Clean Price columns by removing '/year', but still keeping the cureency symbols(₦ and $)
# 1. Define the 'junk' suffixes common in Nigerian listings
# This looks for /year, /annum, or 'per annum' (case insensitive)
removals = r'/(year|annum|month|sqm|day)|\s*per\s*annum'

# 2. Replace the suffixes with an empty string
df['Price_Cleaned'] = (
    df['Price']
    .str.replace(removals, '', regex=True, case=False)
    .str.strip()
)

print(df[['Price', 'Price_Cleaned']].head())

               Price Price_Cleaned
0  ₦ 35,000,000/year  ₦ 35,000,000
1  ₦ 40,000,000/year  ₦ 40,000,000
2  ₦ 13,000,000/year  ₦ 13,000,000
3  ₦ 18,000,000/year  ₦ 18,000,000
4  ₦ 25,000,000/year  ₦ 25,000,000


In [28]:
#Clean Price column, first we convert dollar listings into naira
#Define exchange rate (10th feb, 2026 12:00pm)
usd_to_naira = 1354.97

#Function to convert from dollar to naira
def convert_price(price):
    price = str(price).strip()
    
    # Handle USD listings
    if price.startswith("$"):
        amount_usd = float(price.replace("$", "").replace(",", ""))
        amount_naira = amount_usd * usd_to_naira
    
    # Handle existing Naira listings (cleaning them for consistency)
    else:
        # Remove '₦' and commas, then convert to float
        amount_naira = float(price.replace("₦", "").replace(",", ""))
    
    # Return everything in a consistent ₦ format
    return f"₦{amount_naira:,.2f}"

# Apply to a NEW column
# This keeps 'Price' exactly as it was
df["Price(Converted_to_naira)"] = df["Price_Cleaned"].apply(convert_price)

df[['Price_Cleaned', 'Price(Converted_to_naira)']].head(21)

,Price_Cleaned,Price(Converted_to_naira)
0,"₦ 35,000,000","₦35,000,000.00"
1,"₦ 40,000,000","₦40,000,000.00"
2,"₦ 13,000,000","₦13,000,000.00"
3,"₦ 18,000,000","₦18,000,000.00"
4,"₦ 25,000,000","₦25,000,000.00"
5,"₦ 25,000,000","₦25,000,000.00"
6,"₦ 60,000,000","₦60,000,000.00"
7,"₦ 18,000,000","₦18,000,000.00"
8,"₦ 12,000,000","₦12,000,000.00"
10,"₦ 5,500,000","₦5,500,000.00"


In [34]:
#Convert 'Price(Converted_to_naira)' column into a float , we remove the '₦' symbol and the commas

# Use Regex to keep only digits (\d) and the decimal point (.)
# [^\d.] means "anything that is NOT a digit or a dot"
df["Price(Float)"] = (
    df["Price(Converted_to_naira)"]
    .str.replace(r'[^\d.]', '', regex=True)
    .astype(float)
)

df[["Price(Converted_to_naira)", "Price(Float)"]].head()

,Price(Converted_to_naira),Price(Float)
0,"₦35,000,000.00",35000000.0
1,"₦40,000,000.00",40000000.0
2,"₦13,000,000.00",13000000.0
3,"₦18,000,000.00",18000000.0
4,"₦25,000,000.00",25000000.0


In [35]:
df.head()

,Price,Location,Bedrooms(s),Property Type,Property Category,District,Price_Cleaned,Price(Converted_to_naira),Price(Float)
0,"₦ 35,000,000/year",Diplomatic Zone Asokoro Abuja,5,Furnished Detached Duplex,Duplex,Asokoro,"₦ 35,000,000","₦35,000,000.00",35000000.0
1,"₦ 40,000,000/year",Asokoro District Abuja Diplomatic Zone Asokoro...,6,Mansion With 2room Bq,Duplex,Asokoro,"₦ 40,000,000","₦40,000,000.00",40000000.0
2,"₦ 13,000,000/year",Asokoro Abuja Diplomatic Zone Asokoro Abuja,3,Block Of Flat,Apartment,Asokoro,"₦ 13,000,000","₦13,000,000.00",13000000.0
3,"₦ 18,000,000/year",Guzape District Abuja Diplomatic Zone Guzape A...,4,Terrace Duplex With Bq,Duplex,Guzape,"₦ 18,000,000","₦18,000,000.00",18000000.0
4,"₦ 25,000,000/year","Diplomatic Zone, Guzape Abuja",4,Newly Built Terrace Duplex With Bq,Duplex,Guzape,"₦ 25,000,000","₦25,000,000.00",25000000.0


In [ ]:
BASE_DIR = Path.cwd().parent  
CLEANED = BASE_DIR / "Data" / "cleaned"
df.to_csv(CLEANED / "amac_ppro_rental_cleaned_v1.csv", index=False)


In [37]:
#drop columns
df_new = df.drop(['Price','Location','Property Type','Price_Cleaned', 'Price(Converted_to_naira)'], axis=1)
df_new.head()

,Bedrooms(s),Property Category,District,Price(Float)
0,5,Duplex,Asokoro,35000000.0
1,6,Duplex,Asokoro,40000000.0
2,3,Apartment,Asokoro,13000000.0
3,4,Duplex,Guzape,18000000.0
4,4,Duplex,Guzape,25000000.0


In [ ]:
df_new.rename(columns={'Bedrooms(s)': 'Bedrooms'}, inplace=True)

In [41]:
df_new = df_new[['Bedrooms', 'Property Category', 'Price(Float)', 'District']]
df_new.head()

,Bedrooms,Property Category,Price(Float),District
0,5,Duplex,35000000.0,Asokoro
1,6,Duplex,40000000.0,Asokoro
2,3,Apartment,13000000.0,Asokoro
3,4,Duplex,18000000.0,Guzape
4,4,Duplex,25000000.0,Guzape


In [44]:
df_new.shape

(1888, 4)

In [48]:
df_new.isnull().sum()

Bedrooms             415
Property Category    295
Price(Float)           0
District               0
dtype: int64

In [ ]:
BASE_DIR = Path.cwd().parent  
CLEANED = BASE_DIR / "Data" / "cleaned"
df_new.to_csv(CLEANED / "amac_ppro_rental_cleaned_v2.csv", index=False)